# GrepSeek — interactive demo

[GrepSeek](https://github.com/alirezasalemi7/grepseek) answers questions by **searching a raw Wikipedia corpus with shell commands** (`rg`/`grep`) — *Direct Corpus Interaction* — instead of a dense/sparse index. This notebook spins up the **GRPO** model with vLLM, points the repo's own agent harness at it, and lets you ask questions and watch the agent search.

**Runs on Google Colab *or* a local CUDA box.** Step 0 auto-detects which and adapts: on Colab it installs everything and clones the repo; locally it reuses your existing env, repo, and corpus. Nothing is hard-coded — repo / model / corpus / server are variables you can override with env vars.

> **Requirements**
> - **GPU:** the model is **9B** (bf16 ≈ 18 GB). Best on an **A100 (40 GB)** or **L4 (24 GB)**. On a **T4 (16 GB)** set `SERVE_MODE = "4bit"` in Step 3.
> - **Disk:** the full corpus is **~14 GB** (≈5 GB download). Plain `rg` over 14 GB on CPU takes ~10–30 s, so a query takes ~1 min; the repo's parallel search daemon makes it ms-fast. If disk/speed is tight, use the **sub-corpus** cell (Step 2b).
> - The model + dataset are public on the Hub. On Colab the **code repo must be public** for the Step 1 `git clone` (locally the repo is found automatically).

## Step 0 — Configure (auto-detects Colab vs. local)

Resolves the **repo**, **model**, **corpus**, and **server** — all overridable with env vars (`GREPSEEK_REPO`, `GREPSEEK_MODEL`, `GREPSEEK_CORPUS_ROOT`, `GREPSEEK_HOST`, `GREPSEEK_PORT`) so nothing is hard-coded. E.g. locally point `GREPSEEK_MODEL` at a checkpoint dir to skip the 18 GB download.

In [2]:
import os, sys

# --- detect environment ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# --- locate the repo (Colab: clone target; local: search upward for the repo markers) ---
def _find_repo(start):
    p = os.path.abspath(start)
    while True:
        if all(os.path.exists(os.path.join(p, m)) for m in ('README.md', 'sft', 'rl', 'inference')):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return None
        p = parent

REPO_DIR = os.environ.get('GREPSEEK_REPO') or _find_repo(os.getcwd())
if not REPO_DIR:
    cand = os.path.join(os.getcwd(), 'grepseek')
    REPO_DIR = cand if (IN_COLAB or os.path.isdir(os.path.join(cand, 'inference'))) else os.getcwd()

BASE = os.path.dirname(REPO_DIR)   # /content on Colab, your workspace locally — data/log live here

# --- model: a Hub id works everywhere; override GREPSEEK_MODEL with a local checkpoint dir to skip the download ---
MODEL = os.environ.get('GREPSEEK_MODEL', 'alireza7/GrepSeek-Qwen3.5-9B-GRPO')

# --- corpus + server (all overridable) ---
CORPUS_DIR = os.environ.get('GREPSEEK_CORPUS_ROOT', os.path.join(BASE, 'data', 'wiki_18_corpus'))
HOST = os.environ.get('GREPSEEK_HOST', '127.0.0.1')
PORT = int(os.environ.get('GREPSEEK_PORT', '8000'))

repo_ok   = os.path.isdir(os.path.join(REPO_DIR, 'inference'))
corpus_ok = os.path.exists(os.path.join(CORPUS_DIR, 'wiki_corpus.jsonl'))
print(f"environment : {'Colab' if IN_COLAB else 'local'}")
print(f"repo dir    : {REPO_DIR}  {'(found)' if repo_ok else '(will clone)' if IN_COLAB else '(MISSING — set GREPSEEK_REPO)'}")
print(f"model       : {MODEL}")
print(f"corpus dir  : {CORPUS_DIR}  {'(found)' if corpus_ok else '(will download)'}")
print(f"server      : http://{HOST}:{PORT}")

environment : local
repo dir    : /scratch4/workspace/changzeng_umass_edu-search_and_seek/grepseek  (found)
model       : alireza7/GrepSeek-Qwen3.5-9B-GRPO
corpus dir  : /scratch4/workspace/changzeng_umass_edu-search_and_seek/data/wiki_18_corpus  (found)
server      : http://127.0.0.1:8000


In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU detected — you need a CUDA GPU (A100 40GB / L4 24GB; T4 16GB only with SERVE_MODE="4bit").'
import shutil
print(f'free disk at {BASE}: {shutil.disk_usage(BASE).free/1e9:.0f} GB')

NVIDIA A100-SXM4-80GB, 81920 MiB
free disk at /scratch4/workspace/changzeng_umass_edu-search_and_seek: 175049 GB


## Step 1 — Dependencies & repo

**Colab:** installs `ripgrep`, vLLM 0.17 + the Qwen3.5 transformers build (same recipe as `TRAINING_ENV.md`), and clones the repo. **Local:** skipped — your existing env and repo are reused. Either way the repo goes on `sys.path`.

In [4]:
import os, sys

if IN_COLAB:
    # Colab only: install the serving + Qwen3.5 stack and clone the repo.
    !apt-get -qq update && apt-get -qq install -y ripgrep >/dev/null && rg --version | head -1
    !pip -q install vllm==0.17.0 openai huggingface_hub
    # vLLM 0.17 pulls an OLD huggingface_hub, but the Qwen3.5 transformers build needs is_offline_mode (>=1.x):
    !pip -q install --no-deps 'huggingface_hub==1.8.0' 'tokenizers==0.22.2' 'safetensors==0.7.0'
    !pip -q install --no-deps 'transformers @ git+https://github.com/huggingface/transformers.git@f048e845684894fe60440bb8506f26ffaf7b69ac'
    if not os.path.isdir(os.path.join(REPO_DIR, 'inference')):
        !git clone --depth 1 https://github.com/alirezasalemi7/grepseek {REPO_DIR}
else:
    # Local: reuse your env (ripgrep + vLLM 0.17 + the Qwen3.5 transformers build; see TRAINING_ENV.md).
    print('local: skipping apt/pip/clone — using your existing env and repo.')
    if not os.path.isdir(os.path.join(REPO_DIR, 'inference')):
        print(f'  WARNING: no grepseek repo at {REPO_DIR} — set GREPSEEK_REPO=/path/to/grepseek and re-run Step 0.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('repo on sys.path:', REPO_DIR)

local: skipping apt/pip/clone — using your existing env and repo.
repo on sys.path: /scratch4/workspace/changzeng_umass_edu-search_and_seek/grepseek


## Step 2 — Wikipedia corpus

Downloads `PeterJinGo/wiki-18-corpus` → `wiki_corpus.jsonl` **only if it isn't already at `CORPUS_DIR`** (so locally it's instant). ~5 GB download → ~14 GB on disk.

In [5]:
import os
corpus_file = os.path.join(CORPUS_DIR, 'wiki_corpus.jsonl')
if os.path.exists(corpus_file):
    print('corpus already present — skipping download:', corpus_file)
else:
    os.makedirs(CORPUS_DIR, exist_ok=True)
    print('downloading corpus to', CORPUS_DIR, '...')
    !python {REPO_DIR}/sft/data_generation/download_corpus.py --dest {CORPUS_DIR}
!ls -lh {corpus_file}

corpus already present — skipping download: /scratch4/workspace/changzeng_umass_edu-search_and_seek/data/wiki_18_corpus/wiki_corpus.jsonl
-rw-rwx---+ 1 changzeng_umass_edu changzeng_umass_edu 14G Apr 19 01:19 /scratch4/workspace/changzeng_umass_edu-search_and_seek/data/wiki_18_corpus/wiki_corpus.jsonl


### Step 2b (optional) — sub-corpus fallback

Only if the full corpus is too slow or disk is tight. Keeps the first N passages so `rg` is fast. **Note:** answers to arbitrary questions may not be in a small slice — a quick smoke, not faithful eval. Skip if Step 2 worked.

In [ ]:
# Uncomment to use a sub-corpus instead of the full one, then re-run Steps 4-5.
# import os
# N = 2_000_000   # first ~2M of 21M passages
# SMALL = os.path.join(BASE, 'wiki_18_corpus_small')
# os.makedirs(SMALL, exist_ok=True)
# !head -n {N} {CORPUS_DIR}/wiki_corpus.jsonl > {SMALL}/wiki_corpus.jsonl
# CORPUS_DIR = SMALL
# !ls -lh {CORPUS_DIR}/wiki_corpus.jsonl

## Step 3 — Serve the model with vLLM

Pick a serving mode with **`SERVE_MODE`**:
- **`"bf16"`** (default) — full precision, reuses `rl/serve_rl.sh`. A100 (40 GB) / L4 (24 GB).
- **`"4bit"`** — on-the-fly **bitsandbytes** 4-bit so the 9B model fits a **T4 (16 GB)**. Same Qwen3 reasoning/tool-calling flags as `serve_rl.sh`; it just adds `--quantization bitsandbytes`. Slower (eager), smaller context — demo-quality, not faithful eval.

Launches vLLM in the background and waits until it answers. If `MODEL` is a Hub id, the first run downloads ~18 GB.

In [6]:
import os, sys, subprocess, time, urllib.request, shlex
os.chdir(REPO_DIR)   # serve_rl.sh must run from the repo root

# ── Pick ONE serving mode for your GPU — flip this single variable ───────────
#   "bf16" : full precision. A100 (40 GB) or L4 (24 GB). Reuses rl/serve_rl.sh.
#   "4bit" : on-the-fly bitsandbytes 4-bit (~5-6 GB weights) so 9B fits a T4 (16 GB).
SERVE_MODE = "bf16"
# ─────────────────────────────────────────────────────────────────────────────

LOG = os.path.join(BASE, 'vllm_server.log')
logf = open(LOG, 'w')

if SERVE_MODE == "bf16":
    # Reuse the repo's serve script as-is; just size it for a single GPU.
    env = {**os.environ, 'MODEL_PATH': MODEL, 'TP_SIZE': '1', 'PORT': str(PORT), 'HOST': HOST,
           'GPU_UTIL': '0.92', 'MAX_MODEL_LEN': '16384', 'MAX_NUM_SEQS': '4', 'SERVED_MODEL_NAME': 'grepseek'}
    server = subprocess.Popen(['bash', 'rl/serve_rl.sh'], env=env, stdout=logf, stderr=subprocess.STDOUT)
elif SERVE_MODE == "4bit":
    # serve_rl.sh has no quant knob, so build the SAME vllm command here + bitsandbytes 4-bit.
    try:
        import bitsandbytes  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
    vllm_cmd = [
        'vllm', 'serve', MODEL,
        '--host', HOST, '--port', str(PORT),
        '--served-model-name', 'grepseek',
        '--tensor-parallel-size', '1',
        '--max-model-len', '8192',          # smaller ctx to keep KV cache on a 16 GB T4
        '--max-num-seqs', '2',
        '--gpu-memory-utilization', '0.92',
        '--quantization', 'bitsandbytes',   # <-- the only real difference vs serve_rl.sh
        '--load-format', 'bitsandbytes',
        '--reasoning-parser', 'qwen3',
        '--language-model-only',
        '--trust-remote-code',
        '--enable-auto-tool-choice',
        '--tool-call-parser', 'qwen3_coder',
    ]
    print('Command:', ' '.join(shlex.quote(c) for c in vllm_cmd))
    server = subprocess.Popen(vllm_cmd, env={**os.environ}, stdout=logf, stderr=subprocess.STDOUT)
else:
    raise ValueError(f"SERVE_MODE must be 'bf16' or '4bit', got {SERVE_MODE!r}")

url = f'http://{HOST}:{PORT}/v1/models'
print(f'launched vLLM ({SERVE_MODE}, pid={server.pid}); waiting for {url} ...')
ready = False
for i in range(144):  # up to ~24 min (first run may download ~18 GB if MODEL is a Hub id)
    if server.poll() is not None:
        print('server exited early — log tail:'); print(open(LOG).read()[-3000:]); break
    try:
        urllib.request.urlopen(url, timeout=5); ready = True; break
    except Exception:
        time.sleep(10); print(f'  ...loading ({(i+1)*10}s)', end='\r')
print('\nSERVER READY' if ready else f'\nnot ready — check {LOG}')

launched vLLM (bf16, pid=1290700); waiting for http://127.0.0.1:8000/v1/models ...
  ...loading (170s)
SERVER READY


## Step 4 — Wire the repo's agent harness to the server

We **reuse** `inference.agent.run_agent_on_example` (the exact loop used for the paper's eval): it asks the served model, parses its `<tool_call>`, runs `rg` over `CORPUS_DIR` via the repo's `tools.run_tool`, feeds back `<tool_response>`, and repeats until `<answer>`.

In [7]:
from openai import OpenAI
from transformers import AutoTokenizer
from inference.agent import run_agent_on_example   # reused harness

client = OpenAI(base_url=f'http://{HOST}:{PORT}/v1', api_key='EMPTY')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print('agent ready; corpus =', CORPUS_DIR)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

agent ready; corpus = /scratch4/workspace/changzeng_umass_edu-search_and_seek/data/wiki_18_corpus


## Step 5 — Ask GrepSeek a question

Edit the question and re-run. `ask()` calls the reused harness and pretty-prints the full trajectory (reasoning → shell command → retrieved snippet → answer).

In [8]:
import json

def ask(question, max_turns=6, temperature=0.6):
    rec = run_agent_on_example(
        {'id': 'q', 'question': question},
        client=client, model='grepseek', tokenizer=tokenizer,
        corpus_dir=CORPUS_DIR, max_assistant_turns=max_turns,
        temperature=temperature,
    ).to_dict()
    print('Q:', question, '\n' + '='*80)
    for m in rec['messages']:
        role = m.get('role')
        if role == 'assistant':
            print('\n[assistant]\n' + (m.get('content') or '').strip())
        elif role == 'tool':
            try:
                obj = json.loads(m.get('content') or '{}')
                out = (obj.get('stdout') or '').strip()
                print('  $ ' + (obj.get('command') or ''))
                print('  -> ' + (out[:500] + (' ...[truncated]' if len(out) > 500 else '')))
            except Exception:
                print('  [tool] ' + (m.get('content') or '')[:500])
    print('\n' + '='*80)
    print(f"ANSWER: {rec['prediction']!r}   "
          f"(turns={rec['n_assistant_turns']}, tool_calls={rec['n_tool_calls']}, "
          f"{rec['total_time_s']:.1f}s)")
    return rec

# A two-hop question the released GRPO model answers correctly.
_ = ask('In what country is the second tallest statue in the world?')   # correct answer: China

Q: In what country is the second tallest statue in the world? 

[assistant]
<think>
I need to identify the second tallest statue in the world to determine its location. Since the corpus likely contains a list of the world's tallest statues, I will search for that specific phrase and filter for mentions of "second" to quickly surface the relevant entry.
</think>


<tool_call>
{"name": "shell", "arguments": {"command": "rg -F \"tallest statues in the world\" corpus.jsonl | rg -i -F \"second\" | head -n 8"}}
</tool_call>
  $ rg -F "tallest statues in the world" corpus.jsonl | rg -i -F "second" | head -n 8
  -> 

[assistant]
<think>
The previous search for a specific list of "tallest statues" yielded no results, so I should try a more direct approach by searching for the specific superlative phrase "second tallest statue" instead. This phrasing is likely used in the descriptive text of the relevant article rather than in a list title. I will search the corpus for this exact phrase to see i

### Try your own
GrepSeek is strongest on multi-hop / exact-entity questions. A few to try:

In [9]:
# More two-hop questions the released GRPO model answers correctly:
_ = ask('Who is the father of the father of observational astronomy?')   # correct answer: Vincenzo Galilei
# _ = ask('When was the company that built the first steam locomotive to carry passengers on a public rail line founded?')   # correct answer: 1823
# _ = ask('YOUR QUESTION HERE')

Q: Who is the father of the father of observational astronomy? 

[assistant]
<think>
To answer this question, I first need to identify who is historically recognized as the "father of observational astronomy." I will search the corpus for this specific title to find the relevant figure before proceeding to determine their lineage.
</think>


<tool_call>
{"name": "shell", "arguments": {"command": "rg -F \"father of observational astronomy\" corpus.jsonl | head -n 8"}}
</tool_call>
  $ rg -F "father of observational astronomy" corpus.jsonl | head -n 8
  -> {"id": "8387940", "contents": "\"Galileo Galilei\"\nGalileo Galilei Galileo Galilei (; 15 February 1564 \u2013 8 January 1642) was an Italian astronomer, physicist and engineer, sometimes described as a polymath. Galileo has been called the \"\"father of observational astronomy\"\", the \"\"father of modern physics\"\", the \"\"father of the scientific method\"\", and the \"\"father of modern science\"\". Galileo studied speed and velo

## Notes

- **Use a local checkpoint** (skip the 18 GB download): set `GREPSEEK_MODEL=/path/to/checkpoint` before launching, or edit `MODEL` in Step 0.
- **Speed:** plain `rg` over 14 GB on CPU is the bottleneck (~10–30 s/call). The repo also ships a **sharded-parallel engine + search daemon** (`inference/parallel_search/`) that makes this ms/query — see [`inference/README.md`](https://github.com/alirezasalemi7/grepseek/tree/main/inference).
- **Benchmark eval (EM/F1):** use the repo's `inference/run.py --datasets ...` against the same server instead of `ask()`.
- **SFT model:** set `GREPSEEK_MODEL='alireza7/GrepSeek-Qwen3.5-9B-SFT'` to compare the pre-RL policy.
- **Shut down the server:** `server.terminate()`.

```python
# server.terminate()
```